# H₂ phase estimation with qubitization

Regenerate the H₂ model. Compare the original controlled powers and the reflection-conjugated program by observed energy and depth.

## Distinguish the two construction routes

The optimized root shown below prepares Hartree-Fock data, allocates selector
and phase outputs, and composes the checked `estimate` helper. Its four walk
powers and reflection-conjugation identity are explicit algorithm choices.
FlagQuantum lowers the composition and validates the physical operation budget.
The naive comparison reuses a synthesized controlled walk through Circuit
composition; this is deliberately a different construction route.

Molecular integrals and the full-CI energy are classical preprocessing and
validation. The five signed phase bits become energy through the declared
cosine relation after block-zero postselection. Try a different sampling seed
through the workflow and inspect peak stability, leaving the model and phase
precision fixed. A single favorable energy bin is not evidence of deterministic
success. Compare depth only after both routes are converted to the common CX/U3
basis, not from differently represented source operations.


Use the local Python environment.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Import the FlagQuantum workflow.

In [ ]:
from examples.synthesis.algorithms.quantum_phase_estimation.qpe_with_qubitization.qpe_for_molecule_with_qubitization.overview import (
    model,
    quantum,
    workflow,
)
from examples.synthesis.algorithms.quantum_phase_estimation.qpe_with_qubitization.qpe_for_molecule_with_qubitization.overview.main import (
    print_method,
    print_report,
    save_figure,
)

print(fq.__name__)


The walk uses native signed Pauli selection. Review the controlled block's inputs.

In [ ]:
print(quantum.BLOCK, quantum.DATA, quantum.PHASE)


Use the original H₂ geometry at 0.735 Å.

In [ ]:
print(model.GEOMETRY)


Regenerate the molecular integrals and solve the neutral singlet in STO-3G.

In [ ]:
h2 = model.build_model()
print("RHF energy:", h2.hartree_fock)


Encode the electronic Hamiltonian with all alpha spin orbitals before beta.

In [ ]:
print("Fresh Pauli terms:", len(h2.coefficients))
print("Signed coefficients:", h2.coefficients)


Keep the full CI reference energy.

In [ ]:
print("H2 full CI:", h2.full_ci)


Normalize the Pauli coefficients for LCU state preparation.

In [ ]:
print("LCU scale:", h2.normalization)
print("Selector probabilities:", h2.probabilities)


Use four data bits and a four-bit selector.

In [ ]:
print("Data and selector widths:", len(quantum.DATA), len(quantum.BLOCK))


Inspect the LCU block width and scale.

In [ ]:
print("Block width and scale:", len(quantum.BLOCK), h2.normalization)


Bind the native selector's probabilities and Pauli members.

In [ ]:
print("Native selector fields:", tuple(quantum.bindings(h2)))


Use preparation, signed selection, inverse preparation, and the zero reflection.

In [ ]:
print("Levels in the LCU preparation:", len(quantum.BLOCK))


Interpret each five-bit signed phase through cosine after postselection.

In [ ]:
print("Signed phase extremes:", workflow.decode_phase(16), workflow.decode_phase(15))


Write plots of the joint frequencies retained after selecting block zero.

In [ ]:
figures = Path("notebook-figures")


Prepare the Hartree–Fock state and retain five phase bits.

In [ ]:
print("Phase width:", len(quantum.PHASE))


Synthesize the controlled walk once and compose 31 applications of it.

In [ ]:
print("Controlled walk count:", workflow.PHASE_COUNT - 1)


Run the naïve method and sample the full joint distribution.

In [ ]:
naive = workflow.naive_method(h2)


Read and plot the block-zero counts and the naïve energy.

In [ ]:
print_method("Naive", naive)
save_figure(naive, h2.full_ci, "controlled powers", figures / "naive-histogram.png")
display(Image(filename=str(figures / "naive-histogram.png")))


Conjugate the remaining powers by the conditional reflection.

In [ ]:
print("Uncontrolled walk powers:", (1, 2, 4, 8))


The optimized route uses one checked native qfunc program.

In [ ]:
from typing import Annotated
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, QNum, X, allocate, drop, qfunc
from examples.synthesis.algorithms.quantum_phase_estimation.qpe_with_qubitization.qpe_for_molecule_with_qubitization.operations import (
    estimate,
)


@qfunc
def molecular_phase(
    probabilities: list[float],
    x_members: list[int],
    y_members: list[int],
    z_members: list[int],
    powers: list[int],
    block: Output[Annotated[QNum, 4]],
    phase_var: Output[Annotated[QNum, (5, True, 5)]],
) -> None:
    data: QArray[QBit] = QArray()
    allocate(4, data)
    X(data[0])
    X(data[2])
    allocate(block)
    allocate(phase_var)
    estimate(
        probabilities, x_members, y_members, z_members, powers, block, data, phase_var
    )
    drop(data)


visible_inputs = quantum.bindings(h2)
visible_inputs["powers"] = [1, 2, 4, 8]
visible_program = synth.synthesize(
    molecular_phase,
    classical_inputs=visible_inputs,
    max_qubits=24,
    max_operations=100_000,
    memory_limit_bytes=1024**3,
)
print("Optimized root resources:", visible_program.report["resources"])


Run the reflection-conjugated circuit and sample a second joint distribution.

In [ ]:
optimized = workflow.optimized_method(h2)


Read and plot the optimized route's block-zero outcomes.

In [ ]:
print_method("Optimized", optimized)
save_figure(
    optimized, h2.full_ci, "conjugated powers", figures / "optimized-histogram.png"
)
display(Image(filename=str(figures / "optimized-histogram.png")))


Compare depth in a common CX/U3 basis.

In [ ]:
print("Target depths:", naive.resources.depth, optimized.resources.depth)


Report the most probable block-zero energy for each route.

In [ ]:
print("Two observed energies:", naive.estimate, optimized.estimate)


Compute the original energy-error bound at the five-bit phase resolution.

In [ ]:
tolerance = workflow.error_bound(h2)


Report the bound alongside both sampled results.

In [ ]:
print_report(workflow.Result(h2, naive, optimized, tolerance))


Check both energy-error limits.

In [ ]:
assert abs(naive.estimate - h2.full_ci) < tolerance
assert abs(optimized.estimate - h2.full_ci) < tolerance
